# Feature Visualization

This notebook shows probe scores at each token position.

In [ ]:
from huggingface_hub import login
login(token="")

## Example Experiment

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import torch
import tqdm
import probe_gen.probes as probes
from probe_gen.config import ConfigDict, data
from probe_gen.standard_experiments.hyperparameter_search import get_best_hyperparams_for_train_setup
from probe_gen.standard_experiments.token_heatmaps import visualize_token_heatmap, load_labelled_responses_and_activations, train_probe_on, get_indices_for_each_prediction_type
from probe_gen.gen_data.utils import get_pad_token
from transformers import AutoTokenizer
from probe_gen.config import MODELS

# ========================== Experiment Settings ==========================
probe_type = "mean" 
behaviour = "sycophancy"
datasource = "arguments"
activations_model = "llama_3b"
generation_method = "prompted"
response_model = "llama_3b"
mode = "train"
# =========================================================================

# Train the probe
probe = train_probe_on(probe_type, behaviour, datasource, activations_model, generation_method, response_model, mode, verbose=True)

# Load the labelled responses and activations
responses_df, activations_tensor, attention_mask, labels_tensor = load_labelled_responses_and_activations(
    probe_type, behaviour, datasource, activations_model, generation_method, response_model, mode, verbose=False
)

# Get the tokenizer
model_name = MODELS[activations_model]
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "left"
tokenizer.pad_token = get_pad_token(model_name, tokenizer)

# Get the indices for each prediction type
tp_indices, fp_indices, fn_indices, tn_indices = get_indices_for_each_prediction_type(activations_tensor, attention_mask, labels_tensor, probe)
print(f"True Positives: {len(tp_indices)}")
print(f"False Positives: {len(fp_indices)}")
print(f"False Negatives: {len(fn_indices)}")
print(f"True Negatives: {len(tn_indices)}")

In [ ]:
indices_to_visualize = tp_indices

# Show 10 examples of the given prediction type (e.g. true positives)
for i in range(10):
    visualize_token_heatmap(
        tokenizer,
        indices_to_visualize[i].item(),
        responses_df,
        activations_tensor,
        labels_tensor,
        probe,
        behaviour,
    )

## IGNORE FOR NOW (not token visualisation) - Response Distributions (Arguments)

In [ ]:
from probe_gen.labelling.arguments_autograder import _extract_answer
import matplotlib.pyplot as plt

def plot_response_distributions(responses_df, labels_tensor):
    positive_indices = (labels_tensor == 1).nonzero(as_tuple=True)[0].tolist()
    negative_indices = (labels_tensor == 0).nonzero(as_tuple=True)[0].tolist()

    positive_ratings = []
    for i in range(len(positive_indices)):
        positive_ratings.append(int(_extract_answer(responses_df['model_outputs'][positive_indices[i]])))

    negative_ratings = []
    for i in range(len(negative_indices)):
        negative_ratings.append(int(_extract_answer(responses_df['model_outputs'][negative_indices[i]])))

    bins = np.arange(-0.5, 11.5, 1)
    plt.figure(figsize=(10, 3))
    plt.hist([positive_ratings, negative_ratings], bins=bins, alpha=0.7, label=['Positives', 'Negatives'], edgecolor='black')
    plt.xlabel('Value')
    plt.ylabel('Count')
    plt.title('Histogram Comparison')
    plt.legend()
    plt.xticks(range(0, 11))  # Show integer tick marks
    plt.grid(axis='y', alpha=0.3)
    plt.show()

In [ ]:
from probe_gen.standard_experiments.token_heatmaps import load_labelled_responses_and_activations

probe_type = "mean"
behaviour = "sycophancy"
datasource = "arguments"
activations_model = "llama_3b"
off_policy_model = "qwen_7b"
mode = "train"

for response_strategy in ["on_policy", "incentivised", "prompted", "off_policy"]:
    responses_df, _, _, labels_tensor = load_labelled_responses_and_activations(
        probe_type, behaviour, datasource, activations_model, response_strategy, off_policy_model if response_strategy == "off_policy" else activations_model, mode, verbose=False
    )
    plot_response_distributions(responses_df, labels_tensor)

